# DINO / SAM + PaddleOCR Detection + Hezar CRNN

عکس ← استخراج و تعیین جهت قبلی ← تشخیص نواحی متن با PP-OCRv5_server_det ← خواندن با Hezar CRNN ← جدول شش فیلد.

قبل از اجرا T4 GPU را فعال کنید. سلول اول ZIP کامل GitHub یا ZIP مخصوص پروژه را می‌پذیرد. نصب در دو محیط جدا انجام می‌شود؛ مدل‌ها در اجرای اول دانلود می‌شوند.

Paddle Detection پیش‌فرض CPU است؛ استخراج و خواندن همچنان GPU هستند. این انتخاب صریح است و حالت Paddle GPU در این محیط آزمایش نشده است.

خروجی پیشنهادی است؛ فیلدهای ناموجود خالی و نیازمند بررسی می‌مانند.


## مسیر جدید
DINO + SAM → تعیین جهت → PP-OCRv5_server_det → Hezar CRNN → شش فیلد.

Hezar CRAFT در مسیر پیش‌فرض بارگذاری نمی‌شود. EasyOCR فقط در استخراج/جهت قبلی باقی است.

تشخیص Paddle در محیط و پردازش جدا اجرا می‌شود. پیش‌فرض آن CPU است؛ استخراج کارت و خواندن Hezar روی GPU باقی می‌مانند. برای اجرای Paddle روی GPU، مقدار PADDLE_DEVICE را gpu:0 کنید؛ نصب آن به مخزن رسمی CUDA 12.6 و درایور سازگار نیاز دارد و این مسیر در محیط فعلی آزمایش نشده است.

ZIP کل GitHub یا ZIP مخصوص پروژه هر دو پذیرفته می‌شوند.


In [ ]:
from google.colab import files
from pathlib import Path
import json
import subprocess
import sys
import tempfile
import zipfile

uploaded = files.upload()  # ZIP پروژه، نه عکس کارت
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one dino-sam-hezar-colab.zip project archive')
SOURCE = Path(tempfile.mkdtemp(prefix='persian-id-source-', dir='/content'))
with zipfile.ZipFile(archives[0]) as archive:
    if sum(m.file_size for m in archive.infolist()) > 100 * 1024 * 1024:
        raise ValueError('Source ZIP is unexpectedly large; upload the project ZIP without weights')
    for member in archive.infolist():
        target = (SOURCE / member.filename).resolve()
        if not target.is_relative_to(SOURCE.resolve()):
            raise ValueError('Archive path escapes its destination')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Source ZIP must not contain symbolic links')
    archive.extractall(SOURCE)
projects = [p.parent for p in SOURCE.rglob('pyproject.toml')
            if (p.parent / 'scripts/setup_colab.py').is_file()
            and (p.parent / 'src/persian_id_ocr').is_dir()]
if len(projects) != 1:
    raise RuntimeError('Combined OCR project not found; upload the complete project ZIP')
PROJECT = projects[0]
print('Project:', PROJECT)


## نصب — فقط بار اول
این سلول بسته‌های CUDA و هر دو پکیج محلی را در محیط مجازی نصب می‌کند. کرنل اصلی Colab تغییر نمی‌کند.
وجود GPU بررسی می‌شود؛ این بررسی هنوز به معنی اجرای مدل‌ها روی عکس نیست.

In [ ]:
TEXT_DETECTOR = 'paddle'  # comparison option: craft
PADDLE_DEVICE = 'cpu'  # detection only; DINO/SAM/EasyOCR/Hezar remain on T4 GPU
subprocess.run([sys.executable, str(PROJECT / 'scripts/setup_colab.py'), '--paddle-device', PADDLE_DEVICE], check=True)
PYTHON = '/content/persian-id-venv/bin/python'


## آپلود عکس‌های کارت
یک یا چند عکس انتخاب کنید. جهت به‌صورت خودکار تشخیص داده می‌شود. اگر استخراج نامطمئن باشد، هزار اجرا نمی‌شود و گزارش `needs_review` نمایش داده می‌شود.
برای بررسی دستی جهت می‌توانید `ORIENTATION_OVERRIDE` را تغییر دهید؛ این گزینه چرخش گوشه‌های کارت استخراج‌شده است، نه زاویهٔ عکس خام. هندسهٔ نامعتبر همچنان رد می‌شود.

In [ ]:
photos = files.upload()
INPUT_DIR = Path(tempfile.mkdtemp(prefix='persian-id-inputs-', dir='/content'))
INPUTS = []
for index, (name, content) in enumerate(photos.items()):
    destination = INPUT_DIR / f'{index:03d}_{Path(name).name}'
    destination.write_bytes(content)
    INPUTS.append(str(destination))
if not INPUTS:
    raise ValueError('Upload at least one card photo')
ORIENTATION_OVERRIDE = None  # None (خودکار) یا 0، 90، 180، 270


## اجرای استخراج و OCR
همهٔ عکس‌ها در یک process پردازش می‌شوند و مدل‌ها دوباره برای هر عکس بارگذاری نمی‌شوند. متن نهایی فقط از Hezar است؛ EasyOCR در مرحلهٔ تشخیص جهت استفاده می‌شود.

In [ ]:
OUTPUT_ROOT = Path(tempfile.mkdtemp(prefix='persian-id-results-', dir='/content'))
command = [PYTHON, str(PROJECT / 'scripts/run_batch.py'), *INPUTS,
           '--output-root', str(OUTPUT_ROOT), '--cache-dir', '/content/persian-id-model-cache']
if ORIENTATION_OVERRIDE is not None:
    command += ['--orientation-override', str(ORIENTATION_OVERRIDE)]
command += ['--text-detector', TEXT_DETECTOR, '--paddle-device', PADDLE_DEVICE]
completed = subprocess.run(command)
runs = list(OUTPUT_ROOT.glob('run_*'))
if len(runs) != 1 or not (runs[0] / 'summary.json').is_file():
    raise RuntimeError('Run did not complete; inspect the error above')
RESULTS = runs[0]
SUMMARY = json.loads((RESULTS / 'summary.json').read_text())
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
if completed.returncode == 1:
    print('Runtime error in one or more images. Read the JSON reports below; OCR did not succeed for those images.')
elif completed.returncode == 2:
    print('Some images need review, contain no text, or have no detected card.')
elif completed.returncode != 0:
    raise RuntimeError(f'Unexpected process exit: {completed.returncode}')


In [ ]:
from IPython.display import display, Image as DisplayImage
for item in SUMMARY:
    folder = RESULTS / item['folder']
    report = json.loads((folder / 'result.json').read_text())
    print('\n', item['input'], '→', report['status'])
    for filename in ('card.png', 'card_needs_review.png', 'text_boxes.png', 'field_boxes.png'):
        if (folder / filename).is_file():
            print(filename)
            display(DisplayImage(filename=str(folder / filename)))
    print('متن خام / Raw text:')
    print(report.get('text', ''))
    if report.get('error'):
        print(report['error'])
    if report['status'] in ('needs_review', 'not_found'):
        print(json.dumps(report.get('metadata', {}).get('extraction', {}), ensure_ascii=False, indent=2))

    import pandas as pd
    display(pd.DataFrame(report.get('fields', [])))


## دانلود نتایج
ZIP شامل متن UTF-8، متن نرمال‌شده، تصویر کارت، کادرهای متن و گزارش JSON است. `ocr_candidate` به معنی صحت قطعی نیست. متن خام حفظ می‌شود و فیلدهای هویتی حدس زده نمی‌شوند.

In [ ]:
import shutil
result_zip = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
files.download(result_zip)


## استفاده از تابع استخراج در پروژهٔ خودتان
پس از نصب هر دو پکیج در محیط برنامه:
```python
from cardid import CardExtractor
extractor = CardExtractor()  # یک نمونه برای هر process
card = extractor.extract("photo.jpg")  # تصویر PIL RGB، آماده برای OCR
```
برای استخراج و OCR با هم:
```python
from persian_id_ocr import PersianCardOCR
pipeline = PersianCardOCR(cache_dir="./models")
result = pipeline.process("photo.jpg")
print(result.status, result.text)
```
این مثال‌ها را داخل همان محیط مجازی اجرا کنید؛ پکیج‌ها در کرنل اصلی Colab نصب نشده‌اند. راهنمای بیشتر در README موجود است.